# RedZone world model: one ladder point on a Colab A100 (EXP-107; Saturday = the 150m point)

Personal use only: plan B data, so grades.json carries "personal-use, not for commercial release".
Same portable trainer as the Kaggle TPU notebook (scripts/nfm_train.py: CUDA, bf16, flash attention via SDPA, packing).
Pick the point in POINTS (default 150m, the Saturday run: 156M non-embedding params, 1.2B tokens, ~2.8 h).

Runtime > Change runtime type > **A100 GPU** (High-RAM), then **Run all**. About 3.5 h (~45 CU) for 150m including the
~40 min data build. If the runtime restarts but keeps its disk, Run all again: it resumes from the 30-min checkpoint.


In [ ]:
PIN = "bd1f807883a23294b7905c0343f33e1c77da4cb8"   # kaggle-inputs commit: factory code pack (nfm_train.py + noggin_tokens_v2.py)
REPO = "BouncySlime1215/redzone-everything"
LEAGUES = "nfl,nba,mlb"
LADDER = ["150m"]           # or ["18m", "60m"], ["300m"], ["moe-60m"]
EXPS = {"moe-60m": "EXP-108"}
MINUTES = {"18m": 30, "60m": 80, "150m": 180, "300m": 360, "moe-60m": 100}
CTX, BATCH, TRAIN_TOKENS = 2048, 32, 1.2e9
RUN, CODE = "/content/rz", "/content/rz/code"
ENV = {}
CU_PER_H = {"A100": 13.0, "L4": 5.0, "T4": 2.0}


In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime > Change runtime type > A100 GPU, then Run all again")
GPU = torch.cuda.get_device_name(0)
RATE = next((v for k, v in CU_PER_H.items() if k in GPU), None)
print("GPU:", GPU, "| CU/h:", RATE)


In [ ]:
import json, os, subprocess, sys, tarfile, time, urllib.request
T_START = time.time()
os.makedirs(CODE, exist_ok=True)
if not os.path.exists(f"{CODE}/rz/scripts/nfm_train.py"):
    for i, ref in enumerate((PIN, PIN[:12], PIN, PIN[:12])):     # the raw CDN can lag a fresh push for a minute
        url = f"https://raw.githubusercontent.com/{REPO}/{ref}/factory/rz_factory_code.tar.gz"
        try:
            print("downloading", url)
            urllib.request.urlretrieve(url, f"{RUN}/code.tar.gz")
            break
        except Exception as e:
            print(" ", type(e).__name__, e)
            time.sleep(30 * i)
    else:
        raise SystemExit("could not download the code pack; Run all again in a minute")
    with tarfile.open(f"{RUN}/code.tar.gz") as t:
        t.extractall(CODE, filter="data")
print("code:", sorted(os.listdir(f"{CODE}/rz")))


In [ ]:
def run(preset, extra=()):
    out = f"{RUN}/nfm/{preset}"
    os.makedirs(out, exist_ok=True)
    if os.path.exists(f"{out}/grades.json"):
        print(preset, "done already"); return
    cmd = [sys.executable, "-u", f"{CODE}/rz/scripts/nfm_train.py", "--out", out, "--ckpt-dir", out,
           "--data-dir", f"{RUN}/data", "--build-vocab", "2", "--leagues", LEAGUES, "--preset", preset,
           "--ctx", str(CTX), "--batch", str(BATCH), "--train-tokens", str(TRAIN_TOKENS),
           "--minutes", str(MINUTES[preset]), "--exp", EXPS.get(preset, "EXP-107"), *extra]
    with open(f"{out}/train.log", "a") as log:
        p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                             env={**os.environ, "PYTHONUNBUFFERED": "1", **ENV})
        for line in p.stdout:
            print(line, end=""); log.write(line)
        print(preset, "exit code:", p.wait())

for preset in LADDER:
    run(preset)


In [ ]:
rows = []
for preset in LADDER:
    gp = f"{RUN}/nfm/{preset}/grades.json"
    if os.path.exists(gp):
        g = json.load(open(gp))
        rows.append((preset, g["params"]["non_embedding_active"], g["best_valid_loss"], g["throughput"]["tokens_per_s"],
                     g["throughput"]["tokens_per_s_per_chip"], g["train_minutes"], g["license"]))
print("preset | non-emb params | best valid loss | tok/s | tok/s/chip | train min | license")
for r in rows:
    print(" | ".join(str(x) for x in r))
print(f"session wall clock: {(time.time() - T_START) / 3600:.2f} h")
json.dump(rows, open(f"{RUN}/ladder_summary.json", "w"))
print("CU spent: ~", round((time.time() - T_START) / 3600 * (RATE or 0), 1))
